# Giao diện tạo sinh khai báo (Declarative Generative UI)

Trong bài học này, bạn sẽ định nghĩa một bộ danh mục (catalog) các thành phần giao diện (UI) có thể tái sử dụng và cho phép AI Agent tự động lắp ghép chúng thành các giao diện phong phú — chẳng hạn như bảng điều khiển (dashboard), danh sách chuyến bay,... — mà không cần phải tự lập trình thủ công từng bố cục một.

## 📋 Mục tiêu học tập

1. **Hiểu về đặc tả A2UI (A2UI spec):** Một tiêu chuẩn khai báo giao diện tạo tự động do Google hợp tác xây dựng cùng CopilotKit.
2. **Định nghĩa danh mục thành phần (Component catalogs):** Tạo các định nghĩa và trình xuất (renderers) để Agent có thể lắp ghép trong thời gian thực (runtime).
3. **Sử dụng Lược đồ Động (Dynamic) và Lược đồ Cố định (Fixed):** So sánh bố cục do Agent tự tạo với các mẫu bố cục đã được định nghĩa sẵn.

---

## 🎯 Bạn sẽ xây dựng những gì?

Một giao diện trò chuyện nơi Agent có thể tổng hợp các bề mặt UI phong phú — bảng điều khiển (dashboards), biểu đồ (charts) và thẻ chuyến bay (flight carousels) — từ một danh mục các thành phần có thể tái sử dụng.

> **Prompt mẫu:** "Show me a sales dashboard with total revenue, new customers, and conversion rate metrics." *(Hãy cho tôi xem bảng điều khiển doanh số với các chỉ số tổng doanh thu, khách hàng mới và tỷ lệ chuyển đổi).*

![A2UI Dynamic Dashboard](images/a2ui-dynamic-dashboard.png)

---

## 🧩 Declarative Generative UI là gì?

Giao diện Tạo tự động Khai báo (Declarative Generative UI) cho phép bạn xác định một tập hợp các khối xây dựng UI (UI building blocks) để Agent lắp ghép thành các giao diện. Nó chọn các thành phần từ một danh mục, sắp xếp chúng vào một lược đồ (schema) và liên kết dữ liệu thời gian thực (data bindings) để hiển thị kết quả.

Có 3 phần chính để hệ thống này hoạt động:
* **Danh mục thành phần (Component catalog):** Các nguyên thủy UI mà ứng dụng của bạn hỗ trợ, được chia thành hai phần:
  * **Định nghĩa (Definitions):** Mô tả độc lập với nền tảng về tên, thuộc tính (props) và mục đích của từng thành phần.
  * **Trình xuất (Renderers):** Các triển khai dành riêng cho nền tảng (ví dụ: React component) để biến các định nghĩa thành giao diện thực tế.
* **Lược đồ (Schema):** Mô tả có cấu trúc về việc sử dụng các thành phần nào, cách chúng lồng vào nhau và mối quan hệ giữa chúng.
* **Liên kết dữ liệu (Data bindings):** Các giá trị thời gian thực để điền nội dung thực tế vào lược đồ, chẳng hạn như chi tiết chuyến bay, số liệu hoặc bản ghi.

Bạn có thể liên tưởng mô hình này giống như trò chơi **Lego**: **Danh mục** là hộp chứa các mảnh ghép, **Lược đồ** là cách chúng khớp vào nhau và **Liên kết dữ liệu** là các chi tiết cuối cùng được điền vào. Agent sẽ lắp ráp giao diện một cách linh hoạt trong khi ứng dụng của bạn vẫn giữ toàn quyền kiểm soát về tính nhất quán, an toàn và chất lượng hiển thị.

![A2UI Overview](images/a2ui-overview.png)

---

## ⚖️ Tại sao nên sử dụng Declarative Generative UI?

Controlled Generative UI (Giao diện tạo có kiểm soát - Bài 3) hoạt động rất tốt cho các khu vực có lưu lượng truy cập cao, nơi tính có thể dự đoán là quan trọng nhất. Nhưng đối với "cái đuôi dài" (long tail - như công cụ nội bộ, các trường hợp ngoại lệ, mục tiêu người dùng đa dạng), việc tự lập trình mọi bố cục không thể mở rộng quy mô. Đó là lúc Declarative Generative UI phát huy tác dụng: Agent lắp ráp giao diện từ một tập hợp các khối xây dựng cố định, mang lại cho bạn khả năng thích ứng mà không làm mất đi tính an toàn hoặc nhất quán.

**Ưu điểm (Pros):**
- **Linh hoạt trong khuôn khổ:** Agent thay đổi giao diện nhưng không vượt ra ngoài hệ thống thành phần của bạn.
- **Sử dụng thành phần của riêng bạn (BYOC):** Bạn định nghĩa các nguyên thủy, Agent quyết định cách kết hợp chúng.
- **Tiết kiệm công sức:** Định nghĩa danh mục một lần, tái sử dụng ở mọi nơi.
- **Đa nền tảng từ thiết kế:** Cùng một lược đồ có thể hiển thị trên web, thiết bị di động, Slack hoặc tin nhắn văn bản.
- **Tối ưu Token hơn so với UI tự do:** Agent làm việc với một "từ vựng" cố định thay vì tạo mã nguồn tùy ý.

**Nhược điểm (Cons):**
- **Thiếu kiểm soát chính xác đến từng pixel:** Bạn không thể tinh chỉnh cấu trúc giao diện cuối cùng một cách hoàn hảo tuyệt đối.
- **Khó đoán hơn:** Agent có thể sắp xếp các thành phần khác nhau trong các tình huống tương tự.
- **Dễ gặp lỗi hơn:** Lược đồ và liên kết dữ liệu có thể thất bại theo những cách khó nhận thấy, đòi hỏi logic xác thực và phục hồi.
- **Đòi hỏi thiết kế kỹ từ đầu:** Danh mục thành phần, định dạng lược đồ và cấu trúc của Renderer cần được định nghĩa cẩn thận.

---

## 🛠️ Thiết lập môi trường

Đầu tiên, tải API Key và cài đặt các thư viện cần thiết.

```python
import os
import warnings
warnings.filterwarnings("ignore")

# Load API keys (ví dụ thông qua helper script)
from helper import get_openai_api_key, install_frontend

os.environ["OPENAI_API_KEY"] = get_openai_api_key()
print("✓ OpenAI API key loaded")

# Cài đặt thư viện Frontend
install_frontend()
```

---

## 🤖 1. Xây dựng Agent

[A2UI](https://a2ui.org/) (Agent-to-UI) là một đặc tả do Google tạo ra cho Declarative Generative UI. CopilotKit và AG-UI hỗ trợ A2UI một cách nguyên bản. Trong bài học này, chúng ta sẽ sử dụng A2UI React renderer được duy trì bởi CopilotKit.

### Khởi tạo Server và định nghĩa Agent
Chúng ta sẽ bắt đầu một máy chủ FastAPI với một đồ thị Agent LangGraph. Để kích hoạt tính năng A2UI, bạn cần sử dụng `CopilotKitMiddleware`. Khi A2UI được bật, CopilotKit tự động thêm các hành vi backend cần thiết để tạo đầu ra A2UI có cấu trúc (bạn không cần phải triển khai luồng này thủ công).

Dưới mui xe, việc này hoạt động thông qua hai lớp gọi công cụ (tool calls):
- **Outer tool call:** Kích hoạt khi Agent quyết định tạo A2UI.
- **Inner tool call:** Chứa tải trọng (payload) A2UI có cấu trúc. Middleware chặn các đối số này để cho phép streaming (truyền dữ liệu theo luồng).

Chúng ta cũng sẽ thêm một công cụ `get_sales_data` (truy xuất dữ liệu bán hàng). Điều này cho thấy A2UI hoạt động cùng với các công cụ thông thường: Agent gọi API lấy dữ liệu trước, sau đó trực quan hóa nó bằng `generate_a2ui`.

```python
import json
from fastapi import FastAPI
from copilotkit import CopilotKitMiddleware, LangGraphAGUIAgent
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from helper import start_server

# ── Công cụ lấy dữ liệu (Mock API) ────
@tool
def get_sales_data() -> str:
    """Truy xuất các chỉ số bán hàng và doanh thu hiện tại.
    Trả về dữ liệu bán hàng bao gồm doanh thu, khách hàng, tỷ lệ chuyển đổi, 
    và phân tích theo danh mục và tháng.
    """
    return json.dumps({
        "totalRevenue": "$1.2M",
        "newCustomers": 3842,
        "conversionRate": "3.6%",
        "revenueByCategory": [
            {"label": "Electronics", "value": 420000},
            {"label": "Clothing", "value": 310000},
            {"label": "Home & Garden", "value": 185000},
            {"label": "Sports", "value": 160000},
            {"label": "Books", "value": 125000},
        ],
        "monthlySales": [
            {"label": "Jan", "value": 85000},
            {"label": "Feb", "value": 92000},
            {"label": "Mar", "value": 108000},
            {"label": "Apr", "value": 95000},
            {"label": "May", "value": 115000},
            {"label": "Jun", "value": 125000},
        ],
    })

# ── Tạo đồ thị Agent với CopilotKitMiddleware ────
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[get_sales_data],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "You are a helpful assistant that creates rich visual UI.\n\n"
        "Tool guidance:\n"
        "- For sales/business data requests: first call get_sales_data to fetch "
        "the latest metrics, then call generate_a2ui to visualize the results "
        "as a dashboard with charts, metrics, and cards.\n"
        "- For other rich UI: call generate_a2ui directly.\n\n"
        "IMPORTANT: After calling a tool, do NOT repeat or summarize the data "
        "in your text response. The tool renders UI automatically. "
        "Just confirm what was rendered."
    ),
)

# ── Thiết lập FastAPI Server ────
app = FastAPI()
agent = LangGraphAGUIAgent(
    name="lesson4_agent",
    description="Lesson 4 A2UI agent",
    graph=graph
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")
start_server(app, port=8004)
```

---

## 💻 2. Thêm trình kết xuất A2UI vào Frontend

Frontend có hai phần chính: Điểm cuối (endpoint) **CopilotKit runtime** và **Danh mục thành phần (Component Catalog)**.

### CopilotKit Runtime
Giống như các bài trước, Frontend cần một endpoint runtime để làm cầu nối giữa UI và backend Agent.
Sự khác biệt cốt lõi ở đây là cấu hình `a2ui` — cờ `injectA2UITool: true` báo cho runtime tự động nhúng một công cụ A2UI mà Agent có thể gọi để tạo UI.

*File: `frontend/server.ts`*
```typescript
import { serve } from "@hono/node-server";
import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const langGraphAgent = new LangGraphHttpAgent({ url: "http://localhost:8004" });

const runtime = new CopilotRuntime({
  agents: { default: langGraphAgent },
  a2ui: { injectA2UITool: true }, // Kích hoạt A2UI
});

const app = createCopilotEndpoint({ runtime, basePath: "/api/copilotkit" });

serve({ fetch: app.fetch, port: 4004 }, () => {
  console.log("✓ CopilotKit API server running at http://localhost:4004");
});
```

### Component Catalog — Định nghĩa (Definitions)
Định nghĩa là "hợp đồng" (contract) cho một thành phần trong danh mục của bạn:
- **Tên:** Agent có thể gọi (như `Card`, `Row`, `Text`).
- **Lược đồ (Schema):** Định nghĩa các thuộc tính (`props`) bằng `zod`.
- **Mục đích:** Mô tả bằng ngôn ngữ tự nhiên để Agent hiểu cách dùng.

*File: `frontend/src/catalog/definitions.ts`*
```typescript
import { z } from "zod";

export const demonstrationCatalogDefinitions = {
  Title: {
    description: "A heading. Use for section titles and page headers.",
    props: z.object({
      text: z.string(),
      level: z.string().optional(),
    }),
  },
  Text: {
    description: "A text element. Use for labels, values, captions.",
    props: z.object({
      text: z.union([z.string(), z.object({ path: z.string() })]),
      variant: z.enum(["h1", "h2", "h3", "body", "caption"]).optional(),
    }),
  },
  PieChart: {
    description: "A pie/donut chart. Provide data as array of {label, value, color} objects.",
    props: z.object({
      data: z.array(
        z.object({ label: z.string(), value: z.number(), color: z.string().optional() })
      ),
      innerRadius: z.number().optional(),
    }),
  },
  BarChart: {
    description: "A bar chart. Provide data as array of {label, value} objects.",
    props: z.object({
      data: z.array(z.object({ label: z.string(), value: z.number() })),
      color: z.string().optional(),
    }),
  },
  // ... Các thành phần khác như Card, Row, Column, Metric, Badge, Button...
};

export type DemonstrationCatalogDefinitions = typeof demonstrationCatalogDefinitions;
```

### Component Catalog — Trình xuất (Renderers)
Renderer là nơi chuyển đổi các định nghĩa phía trên thành các thẻ UI thực tế của nền tảng (React Component). Nó ánh xạ một cái tên với một React Node và tự động kiểm tra kiểu (type-check) với Zod.

*File: `frontend/src/catalog/renderers.tsx`*
```tsx
import React from "react";
import { PieChart as RechartsPie, Pie, Cell, ResponsiveContainer, BarChart as RechartsBar, Bar, XAxis, YAxis, Tooltip, CartesianGrid } from "recharts";
import { createCatalog, type CatalogRenderers } from "@copilotkit/a2ui-renderer";
import { demonstrationCatalogDefinitions, type DemonstrationCatalogDefinitions } from "./definitions";

function resolveText(value: unknown): string {
  if (typeof value === "string") return value;
  if (value && typeof value === "object" && "path" in value)
    return String((value as { path: string }).path);
  return String(value ?? "");
}

const demonstrationCatalogRenderers: CatalogRenderers<DemonstrationCatalogDefinitions> = {
  Title: ({ props }) => { /* Render logic for Title */ return <h2>{resolveText(props.text)}</h2>; },
  Text: ({ props }) => { /* Render logic for Text */ return <span>{resolveText(props.text)}</span>; },
  Metric: ({ props }) => { /* Render logic for Metric blocks */ return <div>...</div>; },
  DashboardCard: ({ props, children }) => (
    <div style={{ background: "#fff", padding: "20px", borderRadius: "12px", border: "1px solid #e5e7eb" }}>
      <h4>{resolveText(props.title)}</h4>
      {typeof props.child === "string" && children(props.child)}
    </div>
  ),
  PieChart: ({ props }) => (
    <ResponsiveContainer width="100%" height={200}>
      <RechartsPie>
         {/* Recharts logic */}
      </RechartsPie>
    </ResponsiveContainer>
  ),
  // ... Cài đặt cho các thành phần còn lại
};

// Khởi tạo Catalog hoàn chỉnh
export const demonstrationCatalog = createCatalog(
  demonstrationCatalogDefinitions,
  demonstrationCatalogRenderers,
  {
    catalogId: "copilotkit://app-dashboard-catalog",
    includeBasicCatalog: false,
  },
);
```

### Kết nối mọi thứ vào Ứng dụng React
Trong `main.tsx`, `CopilotKitProvider` sẽ đăng ký danh mục này với tham số `a2ui={{ catalog: demonstrationCatalog }}`.

*File: `frontend/src/main.tsx`*
```tsx
import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import { demonstrationCatalog } from "./catalog/renderers";
import App from "./App";
import "@copilotkit/react-core/v2/styles.css";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit
        useSingleEndpoint={false}
        runtimeUrl="/api/copilotkit"
        // 🪁 Kích hoạt A2UI với danh mục thành phần vừa tạo
        a2ui={{ catalog: demonstrationCatalog }}
      >
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);
```

---

## ⚡ 3. Chạy Ứng dụng: Bố cục Lược đồ Động (Dynamic Schema)

Với mã trên, bạn đã triển khai thành công một hệ thống **Dynamic schema**. Khi người dùng gửi yêu cầu: *"Show me a sales dashboard..."*, Agent sẽ tự động:
1. Gọi `get_sales_data` để lấy cục JSON.
2. Quyết định tự động tạo ra một Dashboard bao gồm `Row`, `Column`, `DashboardCard`, `Metric`, `PieChart`, v.v. bằng công cụ A2UI.

*(Bản xem trước giao diện động (Dynamic Schema): AI hoàn toàn tự nghĩ ra bố cục UI).*
![A2UI Dynamic Schema](images/a2ui-dynamic-dashboard.png)

---

## 🔒 4. Lược đồ Cố định (Fixed Schema) với A2UI Composer

Bên cạnh cách để AI tự do sáng tạo UI (Dynamic Schema), còn một phương pháp khác: **Fixed Schema (Lược đồ cố định)**.

Với Lược đồ cố định, bạn thiết kế sẵn cây thành phần A2UI — bố cục, cách lồng ghép và các biến dữ liệu ràng buộc (data bindings) đã được xác định trước. Nhiệm vụ duy nhất của Agent là gọi API lấy dữ liệu thực và đẩy vào UI.
Điều này cung cấp cho bạn sự kiểm soát tối đa đối với các giao diện cần sự đồng nhất, chỉn chu (ví dụ: giao diện danh sách chuyến bay, hóa đơn).

### Sử dụng A2UI Composer
Cách dễ nhất để xây dựng một lược đồ cố định là sử dụng công cụ [A2UI Composer](https://a2ui-editor.ag-ui.com/). 
Trong Composer, bạn có thể gửi prompt mô tả cấu trúc tĩnh, ví dụ: 
*"Create a carousel of flight cards with origin, destination, duration, time of departure, and time of arrival"*

![A2UI Composer Create](images/a2ui-composer.png)

Composer sẽ sinh ra cho bạn một bản thiết kế (JSON Schema). Bạn có thể sao chép chuỗi JSON này để tích hợp thẳng vào code.

![A2UI Composer Schema](images/a2ui-composer-schema.png)

---

## ✈️ 5. Tích hợp Lược đồ Cố định vào Agent

Trong hệ sinh thái AG-UI, **bất kỳ công cụ nào** cũng có thể trả về các hoạt động A2UI. Bạn chỉ cần gói kết quả trong một mảng `a2ui_operations`. Middleware sẽ tự động phát hiện và gửi xuống Frontend để kết xuất.

Chúng ta sẽ định nghĩa một công cụ `display_flights` trả về một cây thành phần A2UI **đã được định nghĩa sẵn** từ Composer. Agent sẽ gọi `search_flights` để lấy data, rồi ném vào `display_flights`.

```python
import logging; logging.getLogger("langgraph.checkpoint.serde.jsonplus").setLevel(logging.ERROR)
from typing_extensions import TypedDict
from copilotkit import a2ui
from langchain.tools import tool

CATALOG_ID = "copilotkit://app-dashboard-catalog"
SURFACE_ID = "flight-search-results"

# Schema JSON được copy từ A2UI Composer
FLIGHT_SCHEMA = [
    {"id": "root", "component": "List", "children": {"componentId": "flight-card", "path": "/flights"}, "direction": "horizontal", "gap": 16},
    {"id": "flight-card", "component": "Card", "child": "main-col"},
    {"id": "main-col", "component": "Column", "children": ["airline-img", "header-row", "meta-row", "divider-1", "times-row", "route-row", "divider-2", "status-row", "divider-3", "book-btn"], "align": "stretch", "gap": 8},
    # ... Chi tiết các component khác lồng bên trong (Header, Text, Divider, Button) ...
    {"id": "book-btn", "component": "Button", "label": "Book Flight", "variant": "primary", "action": {"event": {"name": "bookFlight"}}},
]

class Flight(TypedDict):
    id: str; airline: str; airlineLogo: str; flightNumber: str; origin: str
    destination: str; date: str; departureTime: str; arrivalTime: str
    duration: str; status: str; price: str
    
# 1. Công cụ tìm kiếm chuyến bay (Data fetching)
@tool
def search_flights(origin: str, destination: str) -> list[Flight]:
    """Tìm kiếm các chuyến bay giữa hai sân bay."""
    return [
        {"id": "1", "airline": "Delta", "airlineLogo": "https://www.gstatic.com/flights/airline_logos/70px/DL.png", "flightNumber": "DL 520", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "08:00", "arrivalTime": "16:35", "duration": "5h 35m", "status": "On Time", "price": "$389"},
        # ... các chuyến bay khác ...
    ]

# 2. Công cụ hiển thị (Gắn kết Dữ liệu vào Lược đồ Cố định)
@tool
def display_flights(flights: list[Flight]) -> str:
    """Hiển thị chuyến bay dưới dạng các thẻ đẹp mắt trên giao diện nằm ngang."""
    return a2ui.render(
        operations=[
            a2ui.create_surface(SURFACE_ID, catalog_id=CATALOG_ID),
            a2ui.update_components(SURFACE_ID, FLIGHT_SCHEMA), # Đẩy Schema tĩnh
            a2ui.update_data_model(SURFACE_ID, {"flights": flights}), # Đẩy Data binding
        ],
    )
```

**Cập nhật lại Agent Graph:**
Chỉ cần thêm các Tool mới vào danh sách và dặn dò Agent trong System Prompt khi nào dùng loại UI nào.

```python
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[get_sales_data, search_flights, display_flights], # Có cả 3 tools
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "You are a helpful assistant that creates rich visual UI.\n\n"
        "Tool guidance:\n"
        "- ALL flight-related queries: first call search_flights to fetch data, "
        "then call display_flights with the results. NEVER use generate_a2ui for flights.\n"
        "- For sales/business requests: call get_sales_data, then generate_a2ui.\n"
        "- For other rich UI: call generate_a2ui directly.\n\n"
    ),
)
```

Giờ đây, khi bạn yêu cầu xem chuyến bay, Agent sẽ áp dụng chính xác bố cục mà bạn đã thiết kế trong Composer.

![A2UI Fixed Schema Flight Carousel](images/a2ui-fixed-carousel.png)

---

## 🆚 So sánh Dynamic Schema vs Fixed Schema: Khi nào dùng cái nào?

| Tiêu chí | **Fixed Schema (Lược đồ Cố định)** | **Dynamic Schema (Lược đồ Động)** |
|---|---|---|
| **Bố cục (Layout)** | Xác định trước, hiển thị giống nhau mỗi lần. | Do Agent tự tạo, thay đổi tùy theo yêu cầu. |
| **Vai trò của Agent** | Chỉ việc điền dữ liệu (Fill data). | Chọn các component và tự sắp xếp bố cục. |
| **Tính nhất quán** | Tối đa tuyệt đối. | Có thể thay đổi / dao động. |
| **Tính linh hoạt** | Rất ít — muốn đổi bố cục phải sửa code/schema. | Rất cao — Agent tự thích ứng với ý định của user. |
| **Phù hợp nhất cho** | Các giao diện trau chuốt, phổ biến, mang tính thương hiệu cao (thẻ chuyến bay, hóa đơn). | Bề mặt nội bộ, linh hoạt, "cái đuôi dài" các trường hợp sử dụng hiếm (long-tail). |

Trong thực tế, nhiều ứng dụng sử dụng kết hợp cả hai: Lược đồ cố định cho các giao diện người dùng chính (high-traffic), và Lược đồ động cho phần còn lại (phân tích dữ liệu ad-hoc, tính năng nội bộ).

---

## 🎓 Tổng kết

- **Declarative Generative UI** cho phép Agent tạo giao diện từ một danh mục các khối xây dựng — linh hoạt hơn UI kiểm soát chặt, nhưng nhất quán hơn UI mở tự do.
- Tiêu chuẩn **A2UI** bao gồm 3 phần: **Component Catalog** (Định nghĩa + Trình xuất), **Schema** (cách tổ chức), và **Data Bindings** (dữ liệu thời gian thực).
- **Dynamic schemas** giao toàn quyền cho AI thiết kế bố cục theo ý muốn — phù hợp với các truy vấn thăm dò dữ liệu.
- **Fixed schemas** giúp bạn làm chủ hoàn toàn giao diện, ép AI chỉ được đẩy dữ liệu vào UI Template sẵn có.
- Cả hai phương pháp có thể song hành cùng nhau trong cùng một AI Agent.

**Bước tiếp theo (Bài 5):**
Trong Bài 5, bạn sẽ tiến đến đầu bên kia của quang phổ: **Giao diện tạo tự do (Open-ended generative UI)**. Bạn sẽ kết nối một ứng dụng MCP (Excalidraw) để Agent có thể chạy toàn bộ ứng dụng lớn ngay trong màn hình chat.